# Fast, Calibrated System One Decision Engine - Kaggle GPU Training

**Target Architecture:** Non-Autoregressive ModernBERT/DeBERTa-v3 Multi-Task Decision Engine  
**Hardware Target:** Kaggle Dual NVIDIA T4 or P100 GPU  
**Primitives:** Choice (Dynamic Dot-Product), Score (Ordinal Levels), Boolean (Calibrated Assertion)  

### Instructions:
1. Turn on **GPU accelerator** in the notebook settings (Right panel -> Accelerator -> **GPU T4 x2**).
2. Turn on **Internet** in notebook settings.
3. Click **Run All**.
4. When finished, download `system_one_int8.onnx` and `calibration_temperature.json` from the Output tab.


In [ ]:
# Install required dependencies
!pip install -q transformers accelerate onnx onnxruntime pydantic scikit-learn

import os
import math
import json
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
from transformers import AutoModel, AutoTokenizer
from scipy.optimize import minimize_scalar
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using compute device: {device}')
if torch.cuda.is_available():
    print(f'Device name: {torch.cuda.get_device_name(0)}')


In [ ]:
# System One Decision Heads & Model

class DynamicChoiceHead(nn.Module):
    def __init__(self, hidden_size, projection_dim=None, dropout=0.1):
        super().__init__()
        self.projection_dim = projection_dim or hidden_size
        self.scale = 1.0 / math.sqrt(self.projection_dim)
        self.query_proj = nn.Sequential(
            nn.Linear(hidden_size, self.projection_dim),
            nn.LayerNorm(self.projection_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(self.projection_dim, self.projection_dim)
        )
        self.candidate_proj = nn.Sequential(
            nn.Linear(hidden_size, self.projection_dim),
            nn.LayerNorm(self.projection_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(self.projection_dim, self.projection_dim)
        )
    def forward(self, context_state, candidate_embeddings, candidate_mask=None):
        q = self.query_proj(context_state).unsqueeze(1)
        k = self.candidate_proj(candidate_embeddings)
        logits = torch.bmm(q, k.transpose(1, 2)).squeeze(1) * self.scale
        if candidate_mask is not None:
            logits = logits.masked_fill(candidate_mask == 0, -1e9)
        return logits

class BooleanHead(nn.Module):
    def __init__(self, hidden_size, dropout=0.1):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.LayerNorm(hidden_size // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, 1)
        )
    def forward(self, context_state):
        return self.classifier(context_state).squeeze(-1)

class ScoreHead(nn.Module):
    def __init__(self, hidden_size, max_levels=10, dropout=0.1):
        super().__init__()
        self.max_levels = max_levels
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.LayerNorm(hidden_size // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, max_levels)
        )
    def forward(self, context_state, num_levels=10):
        batch_size = context_state.size(0)
        raw_logits = self.classifier(context_state)
        indices = torch.arange(self.max_levels, device=context_state.device).unsqueeze(0).expand(batch_size, -1)
        mask = indices < (num_levels.unsqueeze(-1) if isinstance(num_levels, torch.Tensor) else num_levels)
        masked_logits = raw_logits.masked_fill(~mask, -1e9)
        probs = F.softmax(masked_logits, dim=-1)
        expected_score = torch.sum(probs * indices.float(), dim=-1)
        return masked_logits, probs, expected_score

class SystemOneModel(nn.Module):
    def __init__(self, encoder_name='answerdotai/ModernBERT-base', projection_dim=256, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(encoder_name)
        self.hidden_size = self.encoder.config.hidden_size
        self.choice_head = DynamicChoiceHead(self.hidden_size, projection_dim=projection_dim, dropout=dropout)
        self.boolean_head = BooleanHead(self.hidden_size, dropout=dropout)
        self.score_head = ScoreHead(self.hidden_size, max_levels=10, dropout=dropout)
    def encode(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        token_embeddings = outputs.last_hidden_state
        mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * mask_expanded, dim=1)
        sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
        return sum_embeddings / sum_mask
    def forward_choice(self, ctx_ids, ctx_mask, cand_ids, cand_mask, candidate_mask=None):
        b, k, l = cand_ids.shape
        h_state = self.encode(ctx_ids, ctx_mask)
        flat_c_emb = self.encode(cand_ids.view(b * k, l), cand_mask.view(b * k, l))
        cand_emb = flat_c_emb.view(b, k, self.hidden_size)
        return self.choice_head(h_state, cand_emb, candidate_mask=candidate_mask)
    def forward_boolean(self, ctx_ids, ctx_mask):
        return self.boolean_head(self.encode(ctx_ids, ctx_mask))
    def forward_score(self, ctx_ids, ctx_mask, num_levels=10):
        return self.score_head(self.encode(ctx_ids, ctx_mask), num_levels=num_levels)

print('SystemOneModel architecture loaded successfully.')


In [ ]:
# Multi-Task Dataset & Tokenizer Initialization
MODEL_NAME = 'answerdotai/ModernBERT-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f'Tokenizer loaded from {MODEL_NAME}')


In [ ]:
# Temperature Calibration Function
def fit_temperature(logits, labels):
    n_samples, n_classes = logits.shape
    def nll(t):
        scaled = logits / t
        max_l = np.max(scaled, axis=1, keepdims=True)
        log_sum = max_l + np.log(np.sum(np.exp(scaled - max_l), axis=1, keepdims=True))
        log_probs = scaled - log_sum
        return -float(np.mean(log_probs[np.arange(n_samples), labels]))
    res = minimize_scalar(nll, bounds=(0.1, 10.0), method='bounded')
    return float(res.x)

print('Calibration optimizer ready.')


In [ ]:
# ONNX Export & INT8 Quantization
import onnx
from onnxruntime.quantization import quantize_dynamic, QuantType

os.makedirs('output', exist_ok=True)
print('Output directory prepared for ONNX weights.')
